# Ingestion coverage — code-review-graph on `crm-prototype`

CRM backend (Node.js / Express, JavaScript only). Fills the team's `Ingestion_Coverage.xlsx` for this project.
**Functions expected** come from the TypeScript compiler 5.8.3 (`ref_javascript.py` → `js-ref/functions.mjs`).

Every run is a new iteration in `results/crm-prototype/runs/iter-NNN_<date>_<time>/` and adds
lines to `results/crm-prototype/runs.csv`; nothing from an earlier run is overwritten.

Run: `./run.sh crm-prototype` (from `notebooks/`), or Run All here and Save As into the iteration folder.

In [ ]:
APP = 'crm-prototype'   # this notebook's project (folder under apps/)
RUN = ''        # iteration folder name; empty = next free iter-NNN (run.sh sets it)

In [ ]:
import sys; sys.path.insert(0, '.')
import pandas as pd
import coverage_lab as C
import ref_javascript
REFERENCES = [ref_javascript]   # this project's languages that get a functions-expected count
pd.set_option('display.max_rows', 300); pd.set_option('display.max_colwidth', 120)
RUN = RUN or C.next_run_id(APP)
OUT = C.run_dir(APP, RUN)   # results/<APP>/runs/<RUN>/
stamp = C.run_stamp(APP)
{**stamp, 'run': RUN}

## 1. Build the graph exactly as Synthetix does
`CRG_DATA_DIR=<dir> code-review-graph build --repo <root> -q`, after removing the repo from the tool's registry.

In [ ]:
db = C.build(APP, OUT)
db

## 2. Sheet 1 — one row per code file
Read? / Why come from the tool's own filter and a re-parse with the same grammar.

In [ ]:
rows, funcs, left_out = C.coverage(APP, db, REFERENCES)
sheet1 = pd.DataFrame([{k: v for k, v in r.__dict__.items() if k != 'expected'} for r in rows])
sheet1.read.value_counts()

### Files left out of sheet 1 (not code)
No language in the tool and not a programming-language extension: docs, licences, images, Dockerfiles, config.

In [ ]:
print(len(left_out), 'files left out')
pd.Series([p.rsplit('.', 1)[-1] if '.' in p.rsplit('/', 1)[-1] else '(no extension)' for p in left_out]).value_counts()

In [ ]:
sheet1[sheet1.read != 'Fully read'][['file', 'language', 'lines', 'read', 'why', 'lines_read', 'notes']]

## 3. Functions: expected (language's own parser) vs found (graph.db)

In [ ]:
matched = C.match_functions(rows, funcs)
matched.groupby(['language', 'status']).size()

In [ ]:
matched[matched.status != 'found']

### Files the reference parser could not fully check
Listed, never silently dropped (a syntax error, or code the reference cannot parse).

In [ ]:
C.could_not_check(rows)

### Unnamed functions (callbacks, lambdas) — not counted as expected
Shown per file with the lines they cover. Their code is still walked by the tool; only a separate row is missing.

In [ ]:
cb = C.callbacks_table(rows)
print(cb.callbacks.sum(), 'unnamed functions,', cb.lines.sum(), 'lines')
cb.sort_values('callbacks', ascending=False)

## 4. Sheet 2 — per language

In [ ]:
sheet2 = C.parsers_table(APP, rows, matched, RUN, REFERENCES)
sheet2

## 5. Sheet 3 — summary (same formulas as the workbook)

In [ ]:
s = sheet1
summary = {'Files': len(s), 'Fully read': (s.read == 'Fully read').sum(), 'Partly read': (s.read == 'Partly read').sum(),
  'Not read': (s.read == 'Not read').sum(), 'Lines': int(s.lines.fillna(0).sum()), 'Lines read': int(s.lines_read.fillna(0).sum()),
  'Not shown to user': ((s.read != 'Fully read') & (s.shown == 'No')).sum()}
summary['% lines read'] = round(100 * summary['Lines read'] / summary['Lines'], 1)
summary

In [ ]:
s[s.read == 'Not read'].why.value_counts()

## 6. Write the workbook

In [ ]:
C.write_workbook(APP, rows, sheet2, OUT)

## 7. Add this run to the repo's run log
`results/<repo>/runs.csv`, append-only: one line per run (per language with a reference parser).

In [ ]:
log = C.log_run(APP, RUN, stamp, summary, sheet2)
pd.read_csv(log)